# Step 05 — Traffic Quality Gate & Corridor Probe Planning


In [34]:

import copy
import json
import pickle
import shutil
from datetime import datetime, timezone

import networkx as nx
import pandas as pd
from IPython.display import display

from routing.config import (
    RESULTS_DIR,
    ORIGIN,
    ORIGINS,
    DESTINATION,
)
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import (
    collect_traffic_snapshot,
    make_routing_graph,
    save_summary,
)
from routing.traffic_quality import (
    plan_corridor_probes,
    audit_traffic_snapshot,
)
from routing.probe_history import (
    load_probe_history,
    update_probe_history,
    save_probe_history,
)

RADIUS_M = 2000
G = load_road_graph(radius_m=RADIUS_M)

assert isinstance(G, nx.MultiDiGraph)
assert G.is_directed()

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("========== NOTEBOOK 05 INITIALIZED ==========")
print("Graph:", type(G).__name__)
print("Nodes:", G.number_of_nodes())
print("Directed edges:", G.number_of_edges())
print("Results directory:", RESULTS_DIR)
print("PASS: Imports and OSM graph loaded")


========== NOTEBOOK 05 INITIALIZED ==========
Graph: MultiDiGraph
Nodes: 2569
Directed edges: 5850
Results directory: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results
PASS: Imports and OSM graph loaded


## 1 — Load and protect the existing Snapshot


In [35]:

snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"

previous_bundle = None
previous_snapshot = None
previous_summary = {}

def _assert_same_topology(base_graph, candidate_graph, label):
    if not isinstance(candidate_graph, nx.MultiDiGraph):
        raise TypeError(f"{label} must be a networkx.MultiDiGraph")

    if set(base_graph.nodes) != set(candidate_graph.nodes):
        raise ValueError(f"{label} node set differs from the current OSM graph")

    if set(base_graph.edges(keys=True)) != set(candidate_graph.edges(keys=True)):
        raise ValueError(f"{label} directed-edge set differs from the current OSM graph")


if snapshot_path.exists():
    backup_path = RESULTS_DIR / (
        "traffic_snapshot_before_merge_"
        + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        + ".pkl"
    )
    shutil.copy2(snapshot_path, backup_path)

    with snapshot_path.open("rb") as f:
        raw_bundle = pickle.load(f)

    # Support both the current bundle format and an older raw-graph file.
    if isinstance(raw_bundle, dict):
        if "snapshot" not in raw_bundle:
            raise ValueError(
                "traffic_snapshot.pkl is a dictionary but has no 'snapshot' key."
            )
        previous_bundle = raw_bundle
        previous_snapshot = raw_bundle["snapshot"]

    elif isinstance(raw_bundle, nx.MultiDiGraph):
        previous_snapshot = raw_bundle
        previous_bundle = {"snapshot": raw_bundle, "summary": {}}

    else:
        raise TypeError(
            "Unsupported traffic_snapshot.pkl format: "
            f"{type(raw_bundle).__name__}"
        )

    _assert_same_topology(G, previous_snapshot, "saved snapshot")
    previous_summary = previous_bundle.get("summary", {})

    print("Existing snapshot loaded:", snapshot_path)
    print("Backup created:", backup_path)
    print("Nodes:", previous_snapshot.number_of_nodes())
    print("Edges:", previous_snapshot.number_of_edges())

else:
    print("No previous snapshot exists. A new one will be created after live collection.")

print("PASS: Snapshot loading/topology validation completed")


Existing snapshot loaded: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl
Backup created: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot_before_merge_20261106_114732.pkl
Nodes: 2569
Edges: 5850
PASS: Snapshot loading/topology validation completed


## 2 — Plan corridor probes


In [36]:

history = load_probe_history()

REQUEST_LIMIT = 12

plan = plan_corridor_probes(
    G,
    max_probes=REQUEST_LIMIT,
    origins=ORIGINS,
    destination=DESTINATION,
    corridor_width_m=350,
    min_length_m=35,
    min_spacing_m=65,
    history=history,
)

plan_df = pd.DataFrame(plan)

print("Maximum corridor probes:", REQUEST_LIMIT)
print("Planned corridor probes:", len(plan))

if not plan_df.empty and "road_class" in plan_df.columns:
    print("\nRoad-class distribution:")
    display(plan_df["road_class"].value_counts().rename("count").to_frame())

display(plan_df)


Maximum corridor probes: 12
Planned corridor probes: 12

Road-class distribution:


,count
road_class,
trunk,8
primary,4


,edge_id,latitude,longitude,length_m,road_class,corridor,corridor_offset_m
0,"(265565434, 109870, 0)",51.495027,-0.182279,72.25,trunk,Gloucester Road,34.0
1,"(20947178, 1691189798, 0)",51.501582,-0.177494,89.35,primary,Royal Albert Hall,77.2
2,"(2360746811, 6280192964, 0)",51.495728,-0.174309,107.16,trunk,South Kensington,7.6
3,"(265565438, 265565434, 0)",51.495122,-0.181017,103.78,trunk,Gloucester Road,76.0
4,"(385312258, 20947178, 0)",51.501568,-0.178813,93.45,primary,Royal Albert Hall,107.2
5,"(6280192964, 3818359076, 0)",51.495869,-0.172541,140.77,trunk,South Kensington,115.2
6,"(5940891836, 1356901661, 0)",51.494993,-0.183865,143.90,trunk,Gloucester Road,100.8
7,"(109670, 1691189798, 0)",51.501654,-0.175713,158.14,primary,Royal Albert Hall,153.8
8,"(5898995362, 2180741854, 0)",51.495479,-0.176346,180.05,trunk,South Kensington,149.0
9,"(267342172, 265565438, 0)",51.495199,-0.179982,40.51,trunk,Gloucester Road,110.7


## 3 — Select route-focused probes



In [37]:

USE_ROUTE_FOCUSED_PROBES = True
BATCH_SIZE = 6

route_probe_path = RESULTS_DIR / "route_probe_candidates.json"

if USE_ROUTE_FOCUSED_PROBES:
    if not route_probe_path.exists():
        raise FileNotFoundError(
            f"{route_probe_path} not found. Run Notebook 06 through its final "
            "route-probe export cell first."
        )

    candidate_records = json.loads(
        route_probe_path.read_text(encoding="utf-8")
    )

    if not candidate_records:
        raise RuntimeError(
            "Notebook 06 exported an empty route-probe list. "
            "Check that A* found a route and that route edges need traffic observations."
        )

    print("Source: Notebook 06 route-focused candidates")
else:
    candidate_records = plan
    print("Source: corridor plan")

usable_statuses = {"red", "yellow", "clear", "closed"}

def _has_usable_observation(attrs):
    return (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and attrs.get("traffic_status") in usable_statuses
    )

pending_probe_records = []
seen_edge_ids = set()
skipped_observed = 0
skipped_previously_queried = 0

for record in candidate_records:
    edge_id = tuple(record["edge_id"])

    if edge_id in seen_edge_ids:
        continue
    seen_edge_ids.add(edge_id)

    if not G.has_edge(*edge_id):
        raise ValueError(f"Candidate edge is not in the current OSM graph: {edge_id}")

    attrs = (
        previous_snapshot[edge_id[0]][edge_id[1]][edge_id[2]]
        if previous_snapshot is not None
        else G[edge_id[0]][edge_id[1]][edge_id[2]]
    )

    if _has_usable_observation(attrs):
        skipped_observed += 1
        continue

    was_directly_queried = attrs.get("traffic_directly_queried") is True
    reason = str(attrs.get("traffic_reason") or "")
    retryable_api_failure = (
        reason.startswith("api_http_")
        or reason == "api_network_error"
    )

    if was_directly_queried and not retryable_api_failure:
        skipped_previously_queried += 1
        continue

    pending_probe_records.append({
        **record,
        "edge_id": edge_id,
    })

selected_probe_records = pending_probe_records[:BATCH_SIZE]
selected_edge_ids = [
    tuple(record["edge_id"])
    for record in selected_probe_records
]

print("Total candidates:", len(candidate_records))
print("Unique candidates:", len(seen_edge_ids))
print("Already observed and skipped:", skipped_observed)
print("Previously queried and skipped:", skipped_previously_queried)
print("Pending candidates:", len(pending_probe_records))
print("Selected for this run:", len(selected_edge_ids))
print("Maximum requests this run:", BATCH_SIZE)

if selected_probe_records:
    display(pd.DataFrame(selected_probe_records))
else:
    print("No pending probes remain. No API request is needed.")

assert len(selected_edge_ids) <= BATCH_SIZE
assert len(selected_edge_ids) == len(set(selected_edge_ids))


Source: Notebook 06 route-focused candidates
Total candidates: 5
Unique candidates: 5
Already observed and skipped: 0
Previously queried and skipped: 2
Pending candidates: 3
Selected for this run: 3
Maximum requests this run: 6


,edge_id,latitude,longitude,length_m,road_class,current_traffic_status,time_is_imputed
0,"(25291778, 5898995362, 0)",51.495059,-0.174919,114.34,unclassified,missing,True
1,"(5898995362, 2360746811, 0)",51.495620,-0.175066,9.61,trunk_link,missing,True
2,"(2360746811, 6280192964, 0)",51.495728,-0.174309,107.16,trunk,missing,True


## 4 — Merge helper: preserve older valid observations



In [39]:

_TRAFFIC_VALUE_FIELDS = {
    "current_speed_kph",
    "free_flow_speed_kph",
    "travel_time_s",
    "free_flow_time_s",
    "time_is_imputed",
    "free_flow_is_imputed",
    "fallback_speed_kph",
    "is_closed",
}

def _is_traffic_attribute(name):
    return name.startswith("traffic_") or name in _TRAFFIC_VALUE_FIELDS


def _is_usable_observation(attrs):
    return (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and attrs.get("traffic_status") in {
            "red", "yellow", "clear", "closed"
        }
    )


def merge_traffic_snapshots(
    base_graph,
    previous_snapshot,
    fresh_snapshot,
    checked_at_utc,
):
    """Merge newly assessed traffic while retaining older valid observations."""

    _assert_same_topology(base_graph, fresh_snapshot, "fresh_snapshot")

    if previous_snapshot is None:
        merged = base_graph.copy()
    else:
        _assert_same_topology(base_graph, previous_snapshot, "previous_snapshot")
        merged = previous_snapshot.copy()

    refreshed_edges = 0

    for u, v, key, fresh_attrs in fresh_snapshot.edges(keys=True, data=True):
        was_assessed = (
            bool(fresh_attrs.get("traffic_sampled"))
            or bool(fresh_attrs.get("traffic_directly_queried"))
            or fresh_attrs.get("traffic_match_source") == "reverse_inferred"
        )

        if not was_assessed:
            continue

        target_attrs = merged[u][v][key]
        old_is_usable = _is_usable_observation(target_attrs)
        fresh_is_usable = _is_usable_observation(fresh_attrs)

        # Replace old traffic fields only when the new observation is valid,
        # or when there is no older valid observation to preserve.
        if fresh_is_usable or not old_is_usable:
            for attr_name in list(target_attrs):
                if _is_traffic_attribute(attr_name):
                    target_attrs.pop(attr_name, None)

            for attr_name, value in fresh_attrs.items():
                if _is_traffic_attribute(attr_name):
                    target_attrs[attr_name] = copy.deepcopy(value)

        else:
            # Keep the older measured speed/cost, but record the failed
            # or unmatched latest attempt separately.
            target_attrs["traffic_last_attempt_reason"] = fresh_attrs.get(
                "traffic_reason"
            )
            target_attrs["traffic_last_attempt_match_reason"] = fresh_attrs.get(
                "traffic_match_reason"
            )
            target_attrs["traffic_last_attempt_matched"] = bool(
                fresh_attrs.get("traffic_matched")
            )

        target_attrs["traffic_last_checked_at_utc"] = checked_at_utc

        if fresh_is_usable:
            target_attrs["traffic_observed_at_utc"] = checked_at_utc

        refreshed_edges += 1

    merged.graph.update(copy.deepcopy(fresh_snapshot.graph))
    merged.graph["traffic_last_merge_refreshed_edges"] = refreshed_edges

    return merged, refreshed_edges


def count_usable_observations(graph):
    usable = 0
    sampled = 0

    for _, _, _, attrs in graph.edges(keys=True, data=True):
        if attrs.get("traffic_sampled"):
            sampled += 1

        if _is_usable_observation(attrs):
            usable += 1

    total = graph.number_of_edges()
    coverage = 100.0 * usable / total if total else 0.0

    return {
        "usable_observed_edges": usable,
        "sampled_directed_edges": sampled,
        "usable_coverage_percent": round(coverage, 3),
    }

print("PASS: Snapshot merge helpers are ready")


PASS: Snapshot merge helpers are ready


## 5 — Collect live traffic or load the previous snapshot



In [40]:

RUN_LIVE_TRAFFIC = True  # Set True when ready to send up to BATCH_SIZE TomTom requests.

fresh_snapshot = None
batch_summary = None
refreshed_edges = 0

if RUN_LIVE_TRAFFIC:
    if not selected_edge_ids:
        raise RuntimeError(
            "There are no pending probe edges. Nothing will be sent to TomTom."
        )

    with TomTomClient() as client:
        fresh_snapshot, batch_summary = collect_traffic_snapshot(
            osm_graph=G,
            client=client,
            max_requests=len(selected_edge_ids),
            selected_edge_ids=selected_edge_ids,
            pause_s=0.5,
            confidence_threshold=0.65,
        )

    checked_at = (
        batch_summary.get("finished_at_utc")
        or datetime.now(timezone.utc).isoformat()
    )

    snapshot, refreshed_edges = merge_traffic_snapshots(
        G,
        previous_snapshot,
        fresh_snapshot,
        checked_at,
    )

    summary = dict(batch_summary)
    cumulative = count_usable_observations(snapshot)

    summary["merged_refreshed_edges"] = refreshed_edges
    summary["cumulative_usable_observed_edges"] = (
        cumulative["usable_observed_edges"]
    )
    summary["cumulative_sampled_directed_edges"] = (
        cumulative["sampled_directed_edges"]
    )
    summary["cumulative_usable_coverage_percent"] = (
        cumulative["usable_coverage_percent"]
    )

    print("========== LIVE COLLECTION COMPLETED ==========")
    print("Requests made:", summary.get("requests_made"))
    print("Request errors:", summary.get("request_errors"))
    print("Newly refreshed directed edges:", refreshed_edges)
    print("Cumulative usable observations:", cumulative["usable_observed_edges"])

else:
    if previous_snapshot is None:
        raise FileNotFoundError(
            "No saved traffic snapshot exists. Set RUN_LIVE_TRAFFIC = True "
            "to create the first snapshot."
        )

    snapshot = previous_snapshot
    summary = dict(previous_summary)

    print("Loaded existing snapshot; no API request was made.")

_assert_same_topology(G, snapshot, "working snapshot")

print("Snapshot nodes:", snapshot.number_of_nodes())
print("Snapshot directed edges:", snapshot.number_of_edges())
print("Cumulative observation counts:", count_usable_observations(snapshot))


========== LIVE COLLECTION COMPLETED ==========
Requests made: 3
Request errors: 0
Newly refreshed directed edges: 3
Cumulative usable observations: 12
Snapshot nodes: 2569
Snapshot directed edges: 5850
Cumulative observation counts: {'usable_observed_edges': 12, 'sampled_directed_edges': 20, 'usable_coverage_percent': 0.205}


## 6 — Inspect the selected probes against the merged Snapshot

In [41]:

probe_rows = []

for item in selected_probe_records:
    edge_id = tuple(item["edge_id"])
    u, v, key = edge_id
    attrs = snapshot[u][v][key]

    probe_rows.append({
        "edge_id": edge_id,
        "road_class": item.get("road_class", attrs.get("highway")),
        "sampled": attrs.get("traffic_sampled", False),
        "directly_queried": attrs.get("traffic_directly_queried", False),
        "matched": attrs.get("traffic_matched", False),
        "match_source": attrs.get("traffic_match_source"),
        "match_reason": attrs.get("traffic_match_reason"),
        "traffic_status": attrs.get("traffic_status", "missing"),
        "traffic_reason": attrs.get("traffic_reason"),
        "current_speed_kph": attrs.get("current_speed_kph"),
        "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
        "confidence": attrs.get("traffic_confidence"),
        "time_is_imputed": attrs.get("time_is_imputed", True),
    })

probe_results_df = pd.DataFrame(probe_rows)
display(probe_results_df)

if RUN_LIVE_TRAFFIC and batch_summary is not None:
    print("Reverse reassignments:",
          batch_summary.get("reverse_direction_reassignments"))
    print("Reverse inferred observations:",
          batch_summary.get("reverse_inferred_observed_edges"))


,edge_id,road_class,sampled,directly_queried,matched,match_source,match_reason,traffic_status,traffic_reason,current_speed_kph,free_flow_speed_kph,confidence,time_is_imputed
0,"(25291778, 5898995362, 0)",unclassified,True,True,False,NaN,midpoint_too_far,missing,geometry_or_heading_mismatch,NaN,NaN,NaN,True
1,"(5898995362, 2360746811, 0)",trunk_link,True,True,False,NaN,short_segment,missing,geometry_or_heading_mismatch,NaN,NaN,NaN,True
2,"(2360746811, 6280192964, 0)",trunk,True,True,True,direct_probe,candidate_match,yellow,observed,19.0,28.0,1.0,False


Reverse reassignments: 0
Reverse inferred observations: 0


## 7 — Update probe history and audit the Snapshot

In [42]:

if RUN_LIVE_TRAFFIC:
    history = update_probe_history(
        fresh_snapshot,
        batch_summary,
        history=history,
    )

    history_path = save_probe_history(history)
    print("Probe history saved:", history_path)
else:
    print("Probe history was not changed because no live collection was run.")

audit = audit_traffic_snapshot(
    G,
    snapshot,
    summary=summary,
)

print("========== TRAFFIC SNAPSHOT AUDIT ==========")
for key, value in audit.items():
    print(f"{key}: {value}")

assert audit["graph_identity_ok"], "Snapshot topology/length audit failed"
assert audit["costs_ok"], "Snapshot contains invalid routing costs"
assert audit["algorithm_smoke_test_ready"], "Snapshot is not structurally ready"

print("PASS: Snapshot passed structural audit")


Probe history saved: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/probe_history.json
========== TRAFFIC SNAPSHOT AUDIT ==========
graph_identity_ok: True
costs_ok: True
base_nodes: 2569
snapshot_nodes: 2569
total_directed_edges: 5850
sampled_directed_edges: 20
directly_queried_directed_edges: 15
matched_edges: 12
observed_edges: 12
direct_observed_edges: 7
reverse_inferred_observed_edges: 5
missing_edges: 5838
imputed_edges: 5838
invalid_status_edges: 0
observed_coverage_percent: 0.205
direct_observed_coverage_percent: 0.12
reverse_observed_coverage_percent: 0.085
algorithm_smoke_test_ready: True
final_traffic_comparison_ready: False
warnings: ['Observed traffic coverage is sparse; most travel times are not measured', 'TomTom geometry point order is not authoritative legal direction proof']
PASS: Snapshot passed structural audit


## 8 — Build routing graphs for Notebook 06

In [43]:

free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow",
)

measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured",
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured",
)

print("Free-flow graph edges:", free_flow_time.number_of_edges())
print("Measured-time graph edges:", measured_time.number_of_edges())
print("Measured-distance graph edges:", measured_distance.number_of_edges())

print("PASS: Routing graphs prepared")


Free-flow graph edges: 5850
Measured-time graph edges: 5850
Measured-distance graph edges: 5850
PASS: Routing graphs prepared


## 9 — Save the merged Snapshot Bundle



In [44]:

if RUN_LIVE_TRAFFIC:
    previous_runs = []
    if previous_bundle is not None:
        previous_runs = list(previous_bundle.get("collection_runs", []))

    previous_runs.append({
        "finished_at_utc": summary.get("finished_at_utc"),
        "requests_made": summary.get("requests_made"),
        "request_errors": summary.get("request_errors"),
        "newly_refreshed_edges": refreshed_edges,
        "cumulative_usable_observed_edges": summary.get(
            "cumulative_usable_observed_edges"
        ),
    })

    traffic_bundle = dict(previous_bundle or {})
    traffic_bundle.update({
        "snapshot": snapshot,
        "summary": summary,
        "plan": selected_probe_records,
        "corridor_plan": plan,
        "selected_edge_ids": selected_edge_ids,
        "collection_runs": previous_runs,
        "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    })

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    temporary_path = snapshot_path.with_suffix(".tmp")

    with temporary_path.open("wb") as f:
        pickle.dump(
            traffic_bundle,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    temporary_path.replace(snapshot_path)
    save_summary(summary)

    print("PASS: Merged snapshot bundle saved:", snapshot_path)
    print("Saved usable observations:",
          count_usable_observations(snapshot)["usable_observed_edges"])
    print("Backup remains available:", backup_path if previous_snapshot is not None else "No previous snapshot")

else:
    traffic_bundle = previous_bundle
    print("No file was rewritten because RUN_LIVE_TRAFFIC is False.")
    print("Current snapshot:", snapshot_path)


PASS: Merged snapshot bundle saved: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot.pkl
Saved usable observations: 12
Backup remains available: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/traffic_snapshot_before_merge_20261106_114732.pkl


## 10 — Optional interactive traffic map check

In [45]:

traffic_widget, ui_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
)

display(traffic_widget)


In [46]:

print("Origin node:", ui_state["origin_node"])
print("Destination node:", ui_state["destination_node"])
print("Last double-click:", ui_state["last_click_latlon"])

assert ui_state["origin_node"] in G
assert ui_state["destination_node"] in G

if ui_state["last_click_latlon"] is None:
    print("Map loaded correctly. No new double-click was made in this session.")
else:
    print("PASS: Interactive map received a double-click.")


Origin node: 109675
Destination node: 26389375
Last double-click: (51.49878593963934, -0.16403500510874697)
PASS: Interactive map received a double-click.
